# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library. The example dataset contains community mental health survey responses including assessment scores, demographic information, and metadata in the FAIR² Croissant format.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()

print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview

Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their field @id's
print('Available record sets in dataset:')
for record_set in dataset.metadata.record_sets:
    print(f"- RecordSet name: {record_set.name}, @id: {record_set.id}")
    print("   Fields (columns):")
    for field in record_set.fields:
        print(f"     - {field.name}, @id: {field.id}")
    print()

## 3. Data Extraction

Load data from specific record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data from each record set into Pandas DataFrames, using @id for everything
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    # Show user feedback for large datasets
    print(f"Loading records for RecordSet @id: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Fields: {df.columns.tolist()}")

# For demonstration, pick the main survey data record set by guessing the first one
main_record_set_id = record_set_ids[0]
print(f"\nMain RecordSet selected: {main_record_set_id}")
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section may include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare for further analysis.

In [ ]:
# Review columns and choose fields for analysis (by @id)
# Let's pick e.g. GAD-7 total score and gender (replace below with actual @id discovered above)
df = dataframes[main_record_set_id]
print('Columns in main DataFrame:', df.columns.tolist())

# For the demonstration, let's try to use fields that are likely in the dataset according to the metadata
# Let's set field ids for GAD-7 and gender if available
gad7_id = None
gender_id = None

# Try to locate likely field IDs
for col in df.columns:
    if 'gad7' in col.lower() and ('total' in col.lower() or 'score' in col.lower()):
        gad7_id = col
    if 'gender' in col.lower():
        gender_id = col

if gad7_id is None or gender_id is None:
    print(f"Could not auto-detect 'gad7' or 'gender' field. Please check the columns and set these variables manually.")
else:
    print(f"Using GAD-7 field '@id': {gad7_id}")
    print(f"Using gender field '@id': {gender_id}")

    # Drop missing values for GAD-7 field
    filtered_df = df.dropna(subset=[gad7_id])
    # Work only with numeric values
    filtered_df = filtered_df[pd.to_numeric(filtered_df[gad7_id], errors='coerce').notnull()]
    filtered_df[gad7_id] = filtered_df[gad7_id].astype(float)

    # Example filter: GAD-7 total score > threshold
    threshold = 10
    above_threshold = filtered_df[filtered_df[gad7_id] > threshold]
    print(f"Filtered records with {gad7_id} > {threshold} (high anxiety levels):")
    print(above_threshold.head())

    # Normalize GAD-7 values
    filtered_df[f"{gad7_id}_normalized"] = (filtered_df[gad7_id] - filtered_df[gad7_id].mean()) / filtered_df[gad7_id].std()
    print(f"\nNormalized {gad7_id} for filtered records:")
    print(filtered_df[[gad7_id, f"{gad7_id}_normalized"]].head())

    # Group by gender and compute mean GAD-7
    if gender_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(gender_id)[gad7_id].mean()
        print(f"\nAverage {gad7_id} by {gender_id}:")
        print(grouped_df)
    else:
        print(f"'{gender_id}' not available in columns; skip groupby example.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset, using the extracted DataFrame and field `@id`s. E.g. plot the distribution of GAD-7 scores, or boxplots by gender.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if gad7_id is not None and gad7_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[gad7_id].dropna().astype(float), kde=True)
    plt.title(f"Distribution of {gad7_id} (GAD-7 total score)")
    plt.xlabel("GAD-7 Total Score")
    plt.ylabel("Count")
    plt.show()

    # Boxplot by gender if available
    if gender_id is not None and gender_id in df.columns:
        plt.figure(figsize=(7,4))
        sns.boxplot(data=df, x=gender_id, y=gad7_id)
        plt.title(f"{gad7_id} per {gender_id}")
        plt.xlabel("Gender")
        plt.ylabel("GAD-7 Total Score")
        plt.show()
else:
    print("GAD-7 or gender field not set; skip visualization.")

## 6. Conclusion

In this notebook, we demonstrated how to access and explore the FAIR² Mental Health Survey Dataset from Kilifi County using the `mlcroissant` Python library.

- We reviewed the dataset's Croissant metadata, discovered available record sets and fields by their `@id`.
- We extracted the main survey data into a Pandas DataFrame, queried and filtered by assessment scores, normalized values, and inspected group-level statistics.
- Visualizations showcased distributions and differences by demographic variables (e.g., gender).

For more complex analytics, users can repeat these patterns with other fields or record sets from the dataset. All entity references used the canonical `@id` ensuring clarity and reproducibility.